In [ ]:
# Install packages
# %pip install -r requirements.txt

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import math

In [ ]:
def code_break():
    print("-----------------------------------------")

def get_ssr(matrix: np.ndarray, estimators: np.array):
    ""

## Estimators

In [ ]:
apple = pd.read_csv("APPLE.csv")
apple.head()

In [ ]:
X = apple[['regprc', 'ecoprc', 'inseason']]
X = sm.add_constant(X)
y = apple['ecolbs']

# ----- Manual Calc ------
X = np.array(X)
XtX = X.T @ X
det_XtX = np.linalg.det(XtX)
display(f"Determinant XtX: {det_XtX}")
inv_XtX = np.linalg.inv(XtX)
print(inv_XtX)

code_break()

Xty = X.T @ y
print(Xty)

code_break()

estimators = inv_XtX @ Xty
print(estimators)

estimators_results = zip(['const', 'x1', 'x2', 'x3'], estimators)
estimators_results = pd.DataFrame(estimators_results)

In [ ]:
model1 = sm.OLS(exog=X, endog=y)
results = model1.fit()

results.summary()

In [ ]:
stats_package_est = pd.DataFrame(results.params)
# comparison_table.reset_index(inplace=True)
stats_package_est.rename(mapper={0:'stats_package'}, inplace=True, axis=1)
stats_package_est

comparison_table = (
    estimators
    .set_index(0)
    .rename(columns={1: "manual"})
    .join(stats_package_est)
)
comparison_table.index.name = None

display(comparison_table)

## Standard Errors

In [ ]:
# Manual Calculation
beta = np.asarray(estimators).reshape(-1)
e = np.asarray(y).reshape(-1) - X @ beta
e_2 = e**2
ssr = e_2.sum()

s_2 = ssr / (X.shape[0] - X.shape[1])

cov_matrix = s_2*(inv_XtX)
print(cov_matrix)

In [ ]:
manual_cov_matrix = pd.DataFrame(
    cov_matrix, 
    index=['const', 'regprc', 'ecoprc', 'inseason'],
    columns=['const', 'regprc', 'ecoprc', 'inseason']
)
display(manual_cov_matrix)

In [ ]:
sp_cov_matrix = results.cov_params()
for i in range(0, 2):
    sp_cov_matrix.rename(mapper={
        'const':'const',
        'x1':'regprc',
        'x2':'ecoprc',
        'x3':'inseason',
    },
    axis=i,
    inplace=True
    )

sp_cov_matrix

## Model 2

In [200]:
X = apple[['ecoprc', 'regprc', 'reglbs', 'age', 'faminc', 'hhsize', 'numlt5']]
X = sm.add_constant(X)
y = apple['ecolbs']

In [ ]:
model2 = sm.OLS(exog=X, endog=y)
results = model2.fit()

results.summary()

## Test, ecolbs >= 0, ecolbs < 0

## HC2 Errors

In [234]:
## Diagonal
X_np = np.asarray(X)
inv_XtX = np.linalg.inv(X_np.T @ X_np)
projection_matrix = X_np @ inv_XtX @ X_np.T
# print(projection_matrix)
# code_break()
# print(projection_matrix.shape)
# code_break()
px_diag = projection_matrix.diagonal()

## residuals
u2 = np.asarray((results.resid)**2)

print(type(px_diag),type(u2))
print(px_diag.shape, u2.shape)

## Omega
omega_weights = np.asarray(results.resid)**2 / (1 - px_diag)

## HC2 Errors
XtX = X.T @ X
inv_XtX = np.linalg.inv(XtX)

hc2_cov = inv_XtX @ (X_np.T * omega_weights) @ X_np @ inv_XtX
code_break()
pd.DataFrame(hc2_cov, columns=model2.exog_names, index=model2.exog_names)

<class 'numpy.ndarray'> <class 'numpy.ndarray'>
(660,) (660,)
-----------------------------------------


,const,ecoprc,regprc,reglbs,age,faminc,hhsize,numlt5
const,0.212772,-0.136302,0.156867,-2.536973e-03,-0.002762,-2.742463e-04,-0.013898,-0.017852
ecoprc,-0.136302,0.317567,-0.465830,-3.140758e-03,0.002669,5.515218e-05,0.017157,0.010413
regprc,0.156867,-0.465830,0.844100,3.079773e-03,-0.005506,2.477389e-04,-0.037203,-0.020476
reglbs,-0.002537,-0.003141,0.003080,2.087790e-03,0.000018,-5.829303e-07,-0.000229,0.000625
age,-0.002762,0.002669,-0.005506,1.790747e-05,0.000076,-2.644274e-06,0.000319,0.000489
faminc,-0.000274,0.000055,0.000248,-5.829303e-07,-0.000003,4.671345e-06,-0.000034,-0.000005
hhsize,-0.013898,0.017157,-0.037203,-2.288426e-04,0.000319,-3.394481e-05,0.005283,-0.003016
numlt5,-0.017852,0.010413,-0.020476,6.245691e-04,0.000489,-4.607283e-06,-0.003016,0.022393


In [236]:
pd.DataFrame(results.cov_HC2, columns=model2.exog_names, index=model2.exog_names)

,const,ecoprc,regprc,reglbs,age,faminc,hhsize,numlt5
const,0.212772,-0.136302,0.156867,-2.536973e-03,-0.002762,-2.742463e-04,-0.013898,-0.017852
ecoprc,-0.136302,0.317567,-0.465830,-3.140758e-03,0.002669,5.515218e-05,0.017157,0.010413
regprc,0.156867,-0.465830,0.844100,3.079773e-03,-0.005506,2.477389e-04,-0.037203,-0.020476
reglbs,-0.002537,-0.003141,0.003080,2.087790e-03,0.000018,-5.829303e-07,-0.000229,0.000625
age,-0.002762,0.002669,-0.005506,1.790747e-05,0.000076,-2.644274e-06,0.000319,0.000489
faminc,-0.000274,0.000055,0.000248,-5.829303e-07,-0.000003,4.671345e-06,-0.000034,-0.000005
hhsize,-0.013898,0.017157,-0.037203,-2.288426e-04,0.000319,-3.394481e-05,0.005283,-0.003016
numlt5,-0.017852,0.010413,-0.020476,6.245691e-04,0.000489,-4.607283e-06,-0.003016,0.022393
